# Automatic segmentation từ ảnh → Detection3D — **backend DA3 (thay COLMAP hoàn toàn)**

Bản này thay **toàn bộ COLMAP** (SfM + undistort + patch_match_stereo + fusion) bằng **Depth Anything 3 (DA3)**:
một lần feed-forward cho ra **depth per-view + pose + intrinsics (pinhole) + point cloud dense**.

> Vì DA3 và COLMAP **cùng quy ước camera** (extrinsics = world→cam, depth = Z trong hệ camera,
> unproject = pinhole nghịch đảo), nên **chỉ front-end nạp dữ liệu thay đổi**. Toàn bộ Bước 2→7
> (Grounding DINO + SAM2, chiếu 2D→3D, gộp instance, dựng `Detection3D`, export, matching) **giữ nguyên**.

**Đầu vào (thay cho COLMAP dense workspace):**

| Biến | Nội dung | Nguồn |
|---|---|---|
| `NPZ_PATH` | `results.npz` của DA3: `depth (N,H,W)`, `extrinsics (N,3,4 world2cam)`, `intrinsics (N,3,3)`, `conf`, `image` | DA3 export `npz` |
| `SCENE_PLY` | point cloud dense (x,y,z) toàn scene | DA3 unproject depth (đã lọc conf) |
| `SELECTED_VIEWS` | list index view để segment ( `[]` = tất cả ) | — |

Ưu điểm so với COLMAP: chạy được với **ít ảnh**, **không cần undistort**, **không cần patch_match_stereo**
(bước nặng nhất), và **cho depth đặc cả trên tường trơn ít texture** — nơi SfM/MVS hay thất bại.


In [ ]:
%pip install -q opencv-python matplotlib pillow transformers timm accelerate torch torchvision ultralytics open3d plyfile scipy

In [ ]:
import os
import sys
import pickle

import numpy as np

sys.path.insert(0, os.path.abspath("src"))

## Cấu hình (điền đường dẫn DA3 của bạn)

Mặc định trỏ vào `cua_sau_pantry` đã chạy DA3 sẵn. Đổi sang scene khác (room2, hall...) bằng cách
đổi `NPZ_PATH` + `SCENE_PLY`. Chạy **2 lần** (indoor rồi outdoor) nếu muốn matching ở Bước 7.

In [ ]:
# --- DA3 output: MỘT lần inference duy nhất (depth+pose+intrinsics+conf+image) ---
NPZ_PATH  = "workspace/cua_sau_pantry/results.npz"

# Scene cloud: để TRỐNG -> dựng trực tiếp từ chính results.npz (cùng 1 inference => cùng frame
# với điểm backproject, và có MÀU THẬT). Chỉ set 1 file .ply nếu muốn dùng cloud ngoài
# (khi đó phải chắc chắn nó cùng frame/scale với npz, nếu không opening sẽ lệch chỗ).
SCENE_PLY = ""
KEEP_REAL_COLOR = True   # export giữ màu ảnh thật cho scene + tô nổi opening (thay vì tô xám)

# index các view (key-frame) muốn segment; [] = dùng tất cả view trong npz
SELECTED_VIEWS = []

IS_OUTDOOR = False          # True khi chạy cho phía outdoor
OUTPUT_DIR = "outputs"
OUTPUT_NAME = "outdoor" if IS_OUTDOOR else "indoor"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# gộp instance cùng vật thể thấy từ nhiều view (centroid gần nhau hơn ngưỡng -> 1 vật thể)
MERGE_DISTANCE = 0.6        # đơn vị scene (mét nếu đã calibrate SCALE)

# lọc depth không hợp lệ khi chiếu ngược 2D->3D
DEPTH_RANGE = (0.05, 30.0)

# hệ số quy đổi sang mét thật. DA3 có thể không metric -> xem "Hiệu chuẩn scale" ở cuối.
SCALE = 1.0

## Bước 1: nạp pose + intrinsics + depth từ DA3 (thay cell đọc COLMAP)

Dựng adapter `Camera`/`ColmapImage` **cùng interface** mà các bước sau mong đợi
(`camera.intrinsics_matrix()`, `image.rotation_matrix()`, `image.tvec`), nhưng lấy dữ liệu từ `results.npz`.
Không cần parse `cameras.bin/images.bin`, không cần `pycolmap`.

In [ ]:
_Z = np.load(NPZ_PATH)
DA3_DEPTH = _Z["depth"].astype(np.float32)                     # (N,H,W) Z-depth trong hệ camera
DA3_EXTR  = _Z["extrinsics"].astype(np.float64)                # (N,3,4) hoặc (N,4,4), world->cam
DA3_INTR  = _Z["intrinsics"].astype(np.float64)               # (N,3,3)
DA3_CONF  = _Z["conf"] if "conf" in _Z.files else None
DA3_IMAGE = _Z["image"] if "image" in _Z.files else None      # (N,H,W,3) uint8 (ảnh đã xử lý, khớp depth)
N_VIEWS = len(DA3_DEPTH)


class Camera:
    """Thay COLMAP Camera. DA3 intrinsics vốn đã là PINHOLE (không distortion)."""
    def __init__(self, cam_id, K, width, height):
        self.id, self.K, self.width, self.height, self.model = cam_id, np.asarray(K), width, height, "PINHOLE"
    def intrinsics_matrix(self):
        return self.K


class ColmapImage:
    """Thay COLMAP Image. R,t theo quy ước world->cam (giống COLMAP: X_cam = R @ X_world + t)."""
    def __init__(self, img_id, R, t, name):
        self.id, self.R, self.t, self.name = img_id, np.asarray(R), np.asarray(t), name
    def rotation_matrix(self):
        return self.R
    @property
    def tvec(self):
        return self.t


cameras, images = {}, {}
for i in range(N_VIEWS):
    H, W = DA3_DEPTH[i].shape
    cameras[i] = Camera(i, DA3_INTR[i], W, H)
    images[i] = ColmapImage(i, DA3_EXTR[i][:3, :3], DA3_EXTR[i][:3, 3], f"view_{i:04d}")

if not SELECTED_VIEWS:
    SELECTED_VIEWS = list(range(N_VIEWS))

selected_images_info = []
for i in SELECTED_VIEWS:
    print(f"  view {i}: {cameras[i].model}, {cameras[i].width}x{cameras[i].height}")
    selected_images_info.append((images[i].name, images[i], cameras[i]))
print(f"{N_VIEWS} view trong npz; segment {len(SELECTED_VIEWS)} view")

## Bước 2: segment 2D (Grounding DINO + SAM2)

Y hệt `door_window_segmentation_in_2D.ipynb`, chỉ khác **đầu vào là ảnh DA3 trong `results.npz`**
(đúng ảnh mà depth tương ứng, **cùng độ phân giải** → không phải resize mask, khớp pixel tuyệt đối).

In [ ]:
import cv2
import torch
from PIL import Image as PILImage
from ultralytics import SAM
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection

TEXT_PROMPT = "door. window."
BOX_THRESHOLD = 0.5
TEXT_THRESHOLD = 0.25
NMS_IOU_THRES = 0.7
CROSS_LABEL_IOU_THRES = 0.7

DEVICE = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)

GDINO_MODEL_ID = "IDEA-Research/grounding-dino-base"
gdino_processor = AutoProcessor.from_pretrained(GDINO_MODEL_ID)
gdino_model = AutoModelForZeroShotObjectDetection.from_pretrained(GDINO_MODEL_ID).to(DEVICE)
segmenter = SAM("sam2.1_b.pt")

In [ ]:
CLASS_COLORS = {"door": (60, 180, 75), "window": (255, 130, 0)}


def canonical_label(text_label):
    l = text_label.lower()
    if "door" in l:
        return "door"
    if "window" in l:
        return "window"
    return None


def _iou(a, b):
    x1, y1 = max(a[0], b[0]), max(a[1], b[1])
    x2, y2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    area_a = (a[2] - a[0]) * (a[3] - a[1]); area_b = (b[2] - b[0]) * (b[3] - b[1])
    return inter / (area_a + area_b - inter + 1e-9)


def detect_boxes(image_rgb):
    pil_image = PILImage.fromarray(image_rgb)
    inputs = gdino_processor(images=pil_image, text=TEXT_PROMPT, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        outputs = gdino_model(**inputs)
    result = gdino_processor.post_process_grounded_object_detection(
        outputs, inputs.input_ids, threshold=BOX_THRESHOLD, text_threshold=TEXT_THRESHOLD,
        target_sizes=[pil_image.size[::-1]])[0]
    boxes, scores, labels = [], [], []
    for box, score, text_label in zip(result["boxes"], result["scores"], result["labels"]):
        label = canonical_label(text_label)
        if label is not None:
            boxes.append(box.tolist()); scores.append(float(score)); labels.append(label)
    if not boxes:
        return np.zeros((0, 4)), np.array([]), np.array([])
    boxes, scores, labels = np.array(boxes), np.array(scores), np.array(labels)
    keep = []
    for label in np.unique(labels):
        idxs = np.where(labels == label)[0]
        xywh = [[x1, y1, x2 - x1, y2 - y1] for x1, y1, x2, y2 in boxes[idxs].tolist()]
        nms_idx = cv2.dnn.NMSBoxes(xywh, scores[idxs].tolist(), score_threshold=0.0, nms_threshold=NMS_IOU_THRES)
        keep.extend(idxs[np.array(nms_idx).flatten()])
    keep = np.array(keep); boxes, scores, labels = boxes[keep], scores[keep], labels[keep]
    order = np.argsort(scores)[::-1]; final = []
    for i in order:
        if all(_iou(boxes[i], boxes[j]) < CROSS_LABEL_IOU_THRES for j in final):
            final.append(i)
    final = np.array(final)
    return boxes[final], scores[final], labels[final]


def segment_rgb(image_rgb, out_dir, name):
    """Như segment_image gốc, nhưng nhận ảnh RGB (mảng numpy từ DA3) thay vì đường dẫn file.
    Trả về mask nhị phân từng instance ở ĐÚNG độ phân giải ảnh = độ phân giải depth."""
    image_bgr = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR)
    overlay = image_bgr.copy(); h, w = image_bgr.shape[:2]
    boxes, scores, labels = detect_boxes(image_rgb)
    instances = []
    if len(boxes) > 0:
        sam_result = segmenter.predict(image_bgr, bboxes=boxes, verbose=False)[0]
        masks = sam_result.masks.data.cpu().numpy() if sam_result.masks is not None else []
        for box, label, conf, mask in zip(boxes, labels, scores, masks):
            color = CLASS_COLORS.get(label, (0, 0, 255)); mask_bool = mask.astype(bool)
            if mask_bool.shape[:2] != (h, w):
                mask_bool = cv2.resize(mask_bool.astype(np.uint8), (w, h), interpolation=cv2.INTER_NEAREST).astype(bool)
            overlay[mask_bool] = (0.5 * np.array(color) + 0.5 * overlay[mask_bool]).astype(np.uint8)
            x1, y1, x2, y2 = box.astype(int)
            cv2.rectangle(overlay, (x1, y1), (x2, y2), color, 2)
            cv2.putText(overlay, f"{label} {conf:.2f}", (x1, max(y1 - 8, 15)), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2, cv2.LINE_AA)
            instances.append({"label": label, "conf": float(conf), "box": box.tolist(), "mask": mask_bool})
    out_path = os.path.join(out_dir, f"{name}_segmented.jpg")
    cv2.imwrite(out_path, overlay)
    return overlay, instances, out_path

In [ ]:
import matplotlib.pyplot as plt

if DA3_IMAGE is None:
    raise ValueError("results.npz không có key 'image' -> chạy lại DA3 export npz để có ảnh đã xử lý.")

per_image_instances = {}   # view_name -> [ {label, conf, box, mask}, ... ]
overlays = []
for name, img, cam in selected_images_info:
    view_idx = img.id
    image_rgb = DA3_IMAGE[view_idx]                 # (H,W,3) uint8, khớp depth[view_idx]
    overlay, instances, out_path = segment_rgb(image_rgb, OUTPUT_DIR, name)
    per_image_instances[name] = instances
    overlays.append((name, overlay))
    print(f"{name} -> {len(instances)} object(s), overlay: {out_path}")
    for inst in instances:
        print(f"    {inst['label']}: conf={inst['conf']:.2f}, box={inst['box']}")

if overlays:
    fig, axes = plt.subplots(1, len(overlays), figsize=(6 * len(overlays), 6))
    if len(overlays) == 1:
        axes = [axes]
    for ax, (name, overlay) in zip(axes, overlays):
        ax.imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB)); ax.set_title(name); ax.axis("off")
    plt.tight_layout(); plt.show()

## Bước 3: chiếu từng mask 2D → 3D bằng depth + pose của DA3

`backproject_mask_to_world` **giữ nguyên** công thức của notebook COLMAP gốc — vì DA3 dùng đúng
quy ước đó: `X_cam = R @ X_world + t`, depth là Z trong hệ camera. Depth lấy trực tiếp từ
`DA3_DEPTH[view_idx]` (thay cho việc đọc `<ten_anh>.geometric.bin` của COLMAP). Mask và depth
**cùng độ phân giải** nên không cần resize.

In [ ]:
def backproject_mask_to_world(mask, depth, camera, image, depth_range=DEPTH_RANGE):
    """Unproject mọi pixel trong mask có depth hợp lệ -> điểm world.
    Quy ước: X_cam = R @ X_world + t  =>  X_world = R.T @ (X_cam - t)."""
    if mask.shape != depth.shape:
        raise ValueError(f"mask shape {mask.shape} != depth shape {depth.shape}")
    valid = mask.astype(bool) & (depth > depth_range[0]) & (depth < depth_range[1])
    ys, xs = np.nonzero(valid)
    if len(xs) == 0:
        return np.zeros((0, 3))
    K = camera.intrinsics_matrix(); fx, fy, cx, cy = K[0, 0], K[1, 1], K[0, 2], K[1, 2]
    d = depth[ys, xs]
    x_cam = (xs - cx) / fx * d
    y_cam = (ys - cy) / fy * d
    points_cam = np.stack([x_cam, y_cam, d], axis=1)
    R = image.rotation_matrix(); t = image.tvec
    return (points_cam - t) @ R          # = R.T @ (points_cam - t)


raw_instances = []
for name, img, cam in selected_images_info:
    depth = DA3_DEPTH[img.id]                          # (H,W) Z-depth từ DA3
    for inst in per_image_instances[name]:
        mask = inst["mask"]
        if mask.shape != depth.shape:                   # phòng khi seg chạy trên ảnh full-res khác
            mask = cv2.resize(mask.astype(np.uint8), (depth.shape[1], depth.shape[0]),
                              interpolation=cv2.INTER_NEAREST).astype(bool)
        points = backproject_mask_to_world(mask, depth, cam, img) * SCALE
        print(f"{name}: {inst['label']} (conf {inst['conf']:.2f}) -> {len(points)} điểm 3D")
        if len(points) == 0:
            continue
        raw_instances.append({"label": inst["label"], "conf": inst["conf"],
                              "source_image": name, "points": points})

## Bước 4: gộp instance của cùng 1 vật thể thấy từ nhiều view (giữ nguyên)

In [ ]:
def merge_instances(instances, merge_distance=MERGE_DISTANCE):
    clusters = []
    for inst in instances:
        centroid = inst["points"].mean(axis=0); match = None
        for c in clusters:
            if c["label"] != inst["label"]:
                continue
            if np.linalg.norm(np.concatenate(c["points_list"]).mean(axis=0) - centroid) < merge_distance:
                match = c; break
        if match is None:
            clusters.append({"label": inst["label"], "points_list": [inst["points"]]})
        else:
            match["points_list"].append(inst["points"])
    return [{"label": c["label"], "points": np.concatenate(c["points_list"])} for c in clusters]


merged_clusters = merge_instances(raw_instances)
print(f"{len(raw_instances)} instance thô -> {len(merged_clusters)} vật thể sau khi gộp")
for c in merged_clusters:
    print(f"  {c['label']}: {len(c['points'])} điểm, centroid={c['points'].mean(axis=0)}")

## Bước 5: dựng `Detection3D` (dùng đúng `plane_fitting` của sgd_alignment)

Trục "lên" + mặt tường ước lượng trên **toàn bộ** point cloud dense của DA3 (`SCENE_PLY`),
y hệt `manual_segmentation.py` — chỉ khác nguồn điểm mỗi opening là **backproject từ DA3**,
thay vì chọn tay trong CloudCompare.

In [ ]:
from plyfile import PlyData

from sgd_alignment.common.types import Detection3D, PointCloud
from sgd_alignment.detection.plane_fitting import estimate_up_vector_manhattan, extract_wall_planes


def orient_walls_outward(walls, pc, is_outdoor, margin=0.10):
    oriented = {}
    for idx, wall in enumerate(walls):
        wall_point = pc.points[wall.inlier_indices].mean(axis=0); normal = wall.normal
        signed = pc.points @ normal - np.dot(normal, wall_point)
        frac_pos = float((signed > margin).mean()); frac_neg = float((signed < -margin).mean())
        positive_side_is_outward = (frac_pos <= frac_neg) if not is_outdoor else (frac_pos > frac_neg)
        if not positive_side_is_outward:
            normal = -normal
        oriented[idx] = normal
    return oriented


def nearest_wall_normal(centroid, walls, oriented_normals):
    if not walls:
        return None
    best_idx = min(range(len(walls)), key=lambda i: abs(walls[i].signed_distance(centroid[None, :])[0]))
    return oriented_normals[best_idx]


def points_to_detection(points, category, up, wall_normal):
    centroid = points.mean(axis=0)
    if wall_normal is not None:
        normal = wall_normal
    else:
        _, _, vt = np.linalg.svd(points - centroid, full_matrices=False); normal = vt[-1]
    v_axis = up - np.dot(up, normal) * normal; v_axis = v_axis / np.linalg.norm(v_axis)
    u_axis = np.cross(v_axis, normal); u_axis = u_axis / np.linalg.norm(u_axis)
    onto_plane = points - np.outer((points - centroid) @ normal, normal)
    centered = onto_plane - centroid
    u = centered @ u_axis; v = centered @ v_axis
    width = float(u.max() - u.min()); height = float(v.max() - v.min())
    center = centroid + ((u.max() + u.min()) / 2) * u_axis + ((v.max() + v.min()) / 2) * v_axis
    return Detection3D(category=category, center=center, u_axis=u_axis, v_axis=v_axis,
                       normal=normal, width=width, height=height)


def build_scene_from_npz(conf_percentile=40, max_points=1_500_000):
    """Dựng scene cloud từ CHÍNH results.npz: unproject mọi view (lọc confidence), lấy màu ảnh thật.
    Vì dùng đúng depth/pose/intrinsics đã nạp ở Bước 1 nên scene và điểm backproject ở CÙNG một frame."""
    pts, cols = [], []
    for i in range(N_VIEWS):
        d = DA3_DEPTH[i]; K = DA3_INTR[i]; fx, fy, cx, cy = K[0, 0], K[1, 1], K[0, 2], K[1, 2]
        conf = DA3_CONF[i] if DA3_CONF is not None else np.ones_like(d)
        thr = np.percentile(conf, conf_percentile)
        vy, vx = np.nonzero(np.isfinite(d) & (d > DEPTH_RANGE[0]) & (d < DEPTH_RANGE[1]) & (conf >= thr))
        dd = d[vy, vx]
        Xc = np.stack([(vx - cx) / fx * dd, (vy - cy) / fy * dd, dd], axis=1)
        pts.append((Xc - DA3_EXTR[i][:3, 3]) @ DA3_EXTR[i][:3, :3])   # cùng công thức backproject
        cols.append(DA3_IMAGE[i][vy, vx])
    pts = np.concatenate(pts).astype(np.float64) * SCALE
    cols = np.concatenate(cols).astype(np.uint8)
    if len(pts) > max_points:
        idx = np.random.RandomState(0).choice(len(pts), max_points, replace=False)
        pts, cols = pts[idx], cols[idx]
    return pts, cols


if SCENE_PLY:
    _v = PlyData.read(SCENE_PLY)["vertex"].data
    scene_points = np.stack([_v["x"], _v["y"], _v["z"]], axis=1).astype(np.float64) * SCALE
    scene_colors = (np.stack([_v["red"], _v["green"], _v["blue"]], axis=1).astype(np.uint8)
                    if "red" in _v.dtype.names else np.full((len(scene_points), 3), 160, np.uint8))
else:
    scene_points, scene_colors = build_scene_from_npz()   # khuyến nghị: cùng frame + màu thật
scene_pc = PointCloud(points=scene_points)

scene_up = estimate_up_vector_manhattan(scene_pc)
walls = extract_wall_planes(scene_pc, up=scene_up)
oriented_normals = orient_walls_outward(walls, scene_pc, IS_OUTDOOR)
print(f"{len(walls)} mặt tường phát hiện trên scene; up={np.round(scene_up,3)}")

detections = []
for c in merged_clusters:
    wall_normal = nearest_wall_normal(c["points"].mean(axis=0), walls, oriented_normals)
    detections.append(points_to_detection(c["points"], c["label"], scene_up, wall_normal))

for d in detections:
    print(f"{d.category}: center={np.round(d.center,3)}, size={d.width:.2f} x {d.height:.2f}")

## Bước 6: xuất kết quả
- `outputs/<name>_detections.pkl`: `list[Detection3D]` -> đưa thẳng vào `sgd_alignment.matching`.
- `outputs/<name>_openings.ply`: scene (xám) + điểm backproject tô màu theo category, kiểm tra bằng CloudCompare.


In [ ]:
detections_path = os.path.join(OUTPUT_DIR, f"{OUTPUT_NAME}_detections.pkl")
with open(detections_path, "wb") as f:
    pickle.dump(detections, f)
print("đã lưu:", detections_path)

In [ ]:
from plyfile import PlyElement

# Màu tô nổi opening trên nền màu-thật (sáng, dễ thấy). door=xanh lá, window=đỏ.
HIGHLIGHT_COLORS = {"door": (0, 255, 0), "window": (255, 0, 0)}


def export_review_ply(path, scene_points, scene_colors, clusters, keep_real_color=True):
    """Xuất ply kiểm tra: scene GIỮ MÀU THẬT (nếu keep_real_color) + điểm opening tô nổi.
    Đặt keep_real_color=False để về kiểu tô xám như notebook COLMAP gốc."""
    bg = scene_colors if keep_real_color else np.full((len(scene_points), 3), 160, np.uint8)
    all_points = [scene_points]; all_colors = [bg.astype(np.uint8)]
    for c in clusters:
        color = np.array(HIGHLIGHT_COLORS.get(c["label"], (255, 0, 255)), dtype=np.uint8)
        all_points.append(c["points"]); all_colors.append(np.tile(color, (len(c["points"]), 1)))
    pts = np.concatenate(all_points); cols = np.concatenate(all_colors).astype(np.uint8)
    vertex = np.zeros(len(pts), dtype=[("x", "f4"), ("y", "f4"), ("z", "f4"),
                                       ("red", "u1"), ("green", "u1"), ("blue", "u1")])
    vertex["x"], vertex["y"], vertex["z"] = pts[:, 0], pts[:, 1], pts[:, 2]
    vertex["red"], vertex["green"], vertex["blue"] = cols[:, 0], cols[:, 1], cols[:, 2]
    PlyData([PlyElement.describe(vertex, "vertex")], text=False).write(path)


openings_ply_path = os.path.join(OUTPUT_DIR, f"{OUTPUT_NAME}_openings.ply")
export_review_ply(openings_ply_path, scene_points, scene_colors, merged_clusters, keep_real_color=KEEP_REAL_COLOR)
print("đã lưu:", openings_ply_path, "(màu thật)" if KEEP_REAL_COLOR else "(xám)")

## Hiệu chuẩn scale (nếu DA3 không metric)

DA3 có thể xuất depth theo **tỉ lệ tương đối** (không phải mét thật). Khi đó `width/height` đúng hình
nhưng sai đơn vị. Cách chuẩn hoá 1 lần:

1. Chạy pipeline với `SCALE = 1.0`, đọc `height` của một vật thể bạn **biết kích thước thật** (vd cửa cao 2.1 m).
2. Đặt `SCALE = kích_thước_thật / height_đo_được` ở cell config, rồi chạy lại.

Vì mọi toạ độ world của DA3 nằm trong **một hệ nhất quán** (một lần inference cho cả scene),
nhân đều `SCALE` cho toàn bộ điểm là một phép đồng dạng hợp lệ — mọi số đo sẽ ra mét thật.

In [ ]:
# Ví dụ hiệu chuẩn: giả sử vật thể đầu tiên là cửa cao thật 2.1 m
# if detections:
#     real_height_m = 2.1
#     measured = detections[0].height
#     print("SCALE gợi ý =", real_height_m / measured, " (đặt vào cell config rồi chạy lại)")

## (Tuỳ chọn) Bước 7: matching indoor ↔ outdoor

Chạy notebook 2 lần (`IS_OUTDOOR=False` rồi `True`, mỗi lần với `NPZ_PATH`/`SCENE_PLY` của phía đó)
để có `indoor_detections.pkl` + `outdoor_detections.pkl`, rồi chạy cell dưới. Cần **>= 3** opening khớp.

In [ ]:
from sgd_alignment.matching.alignment import align_indoor_outdoor

with open(os.path.join(OUTPUT_DIR, "indoor_detections.pkl"), "rb") as f:
    indoor_detections = pickle.load(f)
with open(os.path.join(OUTPUT_DIR, "outdoor_detections.pkl"), "rb") as f:
    outdoor_detections = pickle.load(f)

result = align_indoor_outdoor(indoor_detections, outdoor_detections)
print(f"{len(result.matches)} cặp khớp, residuals: {result.residuals}")
for indoor_idx, outdoor_idx, cost in result.matches:
    print(f"  indoor[{indoor_idx}] <-> outdoor[{outdoor_idx}], cost={cost:.3f}")